# MindForge — Google Colab Training (Free T4 GPU)

This notebook trains the full MindForge pipeline on Colab's **free T4 GPU** (15 GB VRAM).

### What this notebook does (in order):
| Step | What happens | Time estimate |
|------|-------------|---------------|
| 0 | Check GPU | 10 sec |
| 1 | Mount Google Drive | 30 sec |
| 2 | Install dependencies | 3–5 min |
| 3 | Copy data from Drive → Colab | 2–5 min |
| 4 | Preprocess text data (LLM training) | 5–10 min |
| 5 | Preprocess structured data (risk model) | 1 min |
| 6 | Train XGBoost risk model | 2–5 min |
| 7 | Fine-tune Mistral-7B with QLoRA | **1–2 hours** |
| 8 | Save everything to Google Drive | 5 min |

**Total: ~2 hours** (well within Colab's free session limit)

> **Before starting:** Make sure you selected a T4 GPU!
> Go to `Runtime` → `Change runtime type` → `T4 GPU` → Save

## Step 0: Check Your GPU
Run this first to confirm you have a GPU. You should see `Tesla T4` with ~15 GB.

In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "NO GPU — go to Runtime > Change runtime type > T4 GPU")

print(f"\nPyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name   : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM       : {vram_gb:.1f} GB")
    if vram_gb < 14:
        print("WARNING: Less than 14 GB VRAM detected. LLM fine-tuning may crash.")
    else:
        print("All good — T4 GPU ready!")

## Step 1: Mount Google Drive
Your datasets and trained models will be stored on your Google Drive so they survive session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

In [ ]:
import os

# ── EDIT THESE if you want a different folder name on your Drive ──
DRIVE_ROOT      = "/content/drive/MyDrive/MindForge"
DRIVE_DATA_RAW  = f"{DRIVE_ROOT}/data/raw"
DRIVE_MODELS    = f"{DRIVE_ROOT}/models"

# ── Local Colab disk (faster I/O during training) ─────────────
LOCAL_ROOT      = "/content/MindForge"
DATA_RAW        = f"{LOCAL_ROOT}/data/raw"
DATA_PROCESSED  = f"{LOCAL_ROOT}/data/processed"
MODELS_DIR      = f"{LOCAL_ROOT}/models"

for d in [DATA_RAW, DATA_PROCESSED, MODELS_DIR,
          DRIVE_ROOT, DRIVE_DATA_RAW, DRIVE_MODELS]:
    os.makedirs(d, exist_ok=True)

print("Directories ready!")
print(f"  Local data   : {DATA_RAW}")
print(f"  Drive models : {DRIVE_MODELS}")

## IMPORTANT: Upload Your Data to Google Drive

**Do this before running Step 3.** You only need to do this once.

1. Open [drive.google.com](https://drive.google.com) in a new tab
2. Create this folder path: `My Drive / MindForge / data / raw /`
3. Upload these 5 files into that folder:

| File | Size |
|------|------|
| `Alpie-core_medical_psychology_dataset.json` | ~1.2 GB |
| `cleanData.csv` | ~30 MB |
| `Combined Data.csv` | ~31 MB |
| `mental_health_dataset.csv` | ~596 KB |
| `train.csv` | ~4.5 MB |

> The 1.2 GB file will take a few minutes to upload. Start it, then come back.

Once uploaded, run the next cell to copy data to Colab's local disk.

## Step 2: Install Dependencies
This takes 3–5 minutes. Only needed once per session.

In [ ]:
print("Installing Unsloth (the fast fine-tuning library)...")
# colab-new = optimized build for free Colab T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("\nInstalling other dependencies...")
!pip install -q \
    "transformers>=4.40.0" \
    "datasets>=2.18.0" \
    "trl>=0.8.6" \
    "peft>=0.10.0" \
    "accelerate>=0.29.0" \
    "bitsandbytes>=0.43.0" \
    "xgboost>=2.0.0" \
    "shap>=0.44.0" \
    "scikit-learn>=1.4.0" \
    "imbalanced-learn>=0.11.0" \
    "pandas>=2.1.0" \
    "numpy>=1.26.0" \
    "joblib>=1.4.0" \
    "tqdm>=4.66.0" \
    "matplotlib>=3.8.0" \
    "seaborn>=0.13.0"

print("\nAll dependencies installed!")

## Step 3: Copy Data from Drive to Colab

In [ ]:
import shutil

DATASETS = [
    "Alpie-core_medical_psychology_dataset.json",
    "cleanData.csv",
    "Combined Data.csv",
    "mental_health_dataset.csv",
    "train.csv",
]

print("Copying datasets from Google Drive to local Colab disk...")
print("(This avoids slow Drive I/O during training)\n")

all_ok = True
for fname in DATASETS:
    src = f"{DRIVE_DATA_RAW}/{fname}"
    dst = f"{DATA_RAW}/{fname}"
    if os.path.exists(dst):
        size_mb = os.path.getsize(dst) / 1e6
        print(f"  Already here: {fname} ({size_mb:.0f} MB) — skipping")
        continue
    if not os.path.exists(src):
        print(f"  NOT FOUND: {fname}")
        print(f"    → Upload it to Google Drive: {DRIVE_DATA_RAW}/")
        all_ok = False
        continue
    print(f"  Copying {fname}...", end=" ", flush=True)
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f"done ({size_mb:.0f} MB)")

print()
if all_ok:
    print("All datasets ready!")
else:
    print("Some files are missing — upload them to Drive before continuing.")

## Step 4: Preprocess Text Data (for LLM Fine-Tuning)

This converts the raw datasets into the chat format that Mistral-7B understands:
```
<s>[INST] <<SYS>> You are MindForge... <</SYS>>
User question here [/INST] Assistant answer here </s>
```

**T4 note:** We cap at 20k total examples (down from 45k) to keep training under 2 hours.

In [ ]:
import json, random, re
import pandas as pd
from tqdm import tqdm
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────
TRAIN_JSONL = f"{DATA_PROCESSED}/train.jsonl"
VAL_JSONL   = f"{DATA_PROCESSED}/val.jsonl"

# ── Check if already processed ───────────────────────────────
if os.path.exists(TRAIN_JSONL) and os.path.getsize(TRAIN_JSONL) > 1_000_000:
    print(f"Already preprocessed: {TRAIN_JSONL}")
    with open(TRAIN_JSONL) as f:
        n = sum(1 for _ in f)
    print(f"Train examples: {n:,} — skipping preprocessing")
else:
    # ── Text cleaning ─────────────────────────────────────────
    def clean(text: str) -> str:
        text = str(text).strip()
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # remove non-ASCII
        return text.strip()

    def truncate(text: str, max_chars: int) -> str:
        return text[:max_chars] if len(text) > max_chars else text

    SYSTEM_PROMPT = (
        "You are MindForge, a compassionate and knowledgeable mental health AI assistant. "
        "You provide evidence-based psychological support, education, and guidance. "
        "Always remind users to seek professional help for serious concerns. "
        "Be empathetic, non-judgmental, and supportive."
    )

    def format_mistral(system: str, user: str, assistant: str) -> str:
        return (
            f"<s>[INST] <<SYS>>\n{system}\n<</SYS>>\n\n"
            f"{user} [/INST] {assistant} </s>"
        )

    all_examples = []

    # ── Source 1: Psychology JSON (~1.2 GB) ───────────────────
    # T4: cap at 8000 examples (was 20000)
    MAX_PSYCH = 8000
    print(f"Loading psychology JSON (this may take a minute — it's 1.2 GB)...")
    psych_path = f"{DATA_RAW}/Alpie-core_medical_psychology_dataset.json"
    with open(psych_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    records = data if isinstance(data, list) else next((v for v in data.values() if isinstance(v, list)), [])
    print(f"  Loaded {len(records):,} psychology records")

    psych_examples = []
    for rec in tqdm(records, desc="Processing psychology"):
        prompt   = clean(str(rec.get('prompt', '')))
        response = clean(str(rec.get('response', '')))
        cot      = clean(str(rec.get('complex_cot', '')))
        if not prompt or not response:
            continue
        psych_examples.append({'text': format_mistral(SYSTEM_PROMPT, prompt, truncate(response, 800)), 'source': 'psychology_qa'})
        if cot and len(cot) > 50:
            cot_answer = f"[Thinking]\n{truncate(cot, 400)}\n\n[Answer]\n{truncate(response, 500)}"
            psych_examples.append({'text': format_mistral(SYSTEM_PROMPT, prompt, cot_answer), 'source': 'psychology_cot'})

    # Free the big JSON from RAM before we continue
    del data, records
    import gc; gc.collect()

    random.seed(42)
    if len(psych_examples) > MAX_PSYCH:
        psych_examples = random.sample(psych_examples, MAX_PSYCH)
    all_examples.extend(psych_examples)
    print(f"  Psychology examples: {len(psych_examples):,}")

    # ── Source 2: Therapy Q&A ─────────────────────────────────
    MAX_THERAPY = 5000
    print("Loading therapy Q&A...")
    therapy_df = pd.read_csv(f"{DATA_RAW}/mental_health_dataset.csv")
    therapy_examples = []
    for _, row in therapy_df.iterrows():
        ctx  = clean(str(row.iloc[0]))
        resp = clean(str(row.iloc[1]))
        if not ctx or not resp or len(ctx) < 10:
            continue
        therapy_examples.append({'text': format_mistral(SYSTEM_PROMPT, ctx, truncate(resp, 800)), 'source': 'therapy_qa'})
    if len(therapy_examples) > MAX_THERAPY:
        therapy_examples = random.sample(therapy_examples, MAX_THERAPY)
    all_examples.extend(therapy_examples)
    print(f"  Therapy examples: {len(therapy_examples):,}")

    # ── Source 3: Clean Statements ────────────────────────────
    MAX_STATEMENTS = 7000
    print("Loading statement classification data...")
    label_map = {
        'anxiety': 'Anxiety', 'depression': 'Depression', 'normal': 'Normal / No concern',
        'suicidal': 'Suicidal ideation — immediate professional help is needed',
        'stress': 'Stress', 'bipolar': 'Bipolar tendencies',
        'personality disorder': 'Personality disorder indicators',
    }
    stmt_df = pd.read_csv(f"{DATA_RAW}/cleanData.csv")
    stmt_examples = []
    for _, row in stmt_df.iterrows():
        stmt   = clean(str(row.iloc[1] if len(row) > 2 else row.iloc[0]))
        status = label_map.get(str(row.iloc[-1]).strip().lower(), str(row.iloc[-1]).strip().title())
        if not stmt or len(stmt) < 8:
            continue
        user_msg = f'Please analyse this statement and identify any mental health signals: "{stmt}"'
        asst_msg = (
            f"Based on the language and emotional tone, this statement shows signs consistent with: **{status}**.\n\n"
            f"If you or someone you know is struggling, please reach out to a mental health professional."
        )
        stmt_examples.append({'text': format_mistral(SYSTEM_PROMPT, user_msg, asst_msg), 'source': 'classification'})
    if len(stmt_examples) > MAX_STATEMENTS:
        stmt_examples = random.sample(stmt_examples, MAX_STATEMENTS)
    all_examples.extend(stmt_examples)
    print(f"  Statement examples: {len(stmt_examples):,}")

    # ── Shuffle and split ─────────────────────────────────────
    random.shuffle(all_examples)
    split = int(len(all_examples) * 0.95)
    train_data, val_data = all_examples[:split], all_examples[split:]
    print(f"\nTotal: {len(all_examples):,}  |  Train: {len(train_data):,}  |  Val: {len(val_data):,}")

    # ── Save JSONL ────────────────────────────────────────────
    def write_jsonl(records, path):
        with open(path, 'w', encoding='utf-8') as f:
            for rec in tqdm(records, desc=f"Writing {Path(path).name}"):
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    write_jsonl(train_data, TRAIN_JSONL)
    write_jsonl(val_data, VAL_JSONL)
    print(f"\nSaved: {TRAIN_JSONL}")
    print(f"Saved: {VAL_JSONL}")

print("\nText preprocessing complete!")

## Step 5: Preprocess Structured Data (for Risk Model)

In [ ]:
import pandas as pd
import numpy as np
import json

STRUCTURED_CSV   = f"{DATA_PROCESSED}/structured_clean.csv"
LABEL_MAP_JSON   = f"{DATA_PROCESSED}/label_mapping.json"
TARGET           = "mental_health_risk"

if os.path.exists(STRUCTURED_CSV) and os.path.getsize(STRUCTURED_CSV) > 100:
    df_check = pd.read_csv(STRUCTURED_CSV)
    print(f"Already preprocessed: {df_check.shape[0]:,} rows — skipping")
else:
    CATEGORICAL_COLS = ['gender','employment_status','work_environment','mental_health_history','seeks_treatment']
    NUMERIC_COLS     = ['age','stress_level','sleep_hours','physical_activity_days',
                        'depression_score','anxiety_score','social_support_score','productivity_score']

    def norm_cols(df):
        df.columns = df.columns.str.strip().str.lower().str.replace(' ','_').str.replace('-','_')
        return df

    print("Loading structured data...")
    combined = norm_cols(pd.read_csv(f"{DATA_RAW}/Combined Data.csv"))
    train_df = norm_cols(pd.read_csv(f"{DATA_RAW}/train.csv"))

    for frame in [combined, train_df]:
        for alt in ['status','risk_level','label','target']:
            if alt in frame.columns and TARGET not in frame.columns:
                frame.rename(columns={alt: TARGET}, inplace=True)

    df = pd.concat([combined, train_df], ignore_index=True)
    df.drop_duplicates(inplace=True)
    print(f"  Merged: {df.shape[0]:,} rows")

    # Clean
    df.dropna(subset=[TARGET], inplace=True)
    df[TARGET] = df[TARGET].astype(str).str.strip().str.title()

    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col].fillna(df[col].median(), inplace=True)

    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()
            df[col].replace('Nan', np.nan, inplace=True)
            df[col].fillna(df[col].mode()[0], inplace=True)

    # Feature engineering
    if all(c in df.columns for c in ['sleep_hours','physical_activity_days','social_support_score']):
        df['wellness_score'] = (
            df['sleep_hours']/9.0 + df['physical_activity_days']/7.0 + df['social_support_score']/10.0
        ) / 3.0

    if all(c in df.columns for c in ['depression_score','anxiety_score','stress_level']):
        df['distress_index'] = (
            df['depression_score']/30.0 + df['anxiety_score']/21.0 + df['stress_level']/10.0
        ) / 3.0

    if 'age' in df.columns:
        df['age_group'] = pd.cut(df['age'], bins=[0,18,25,35,50,65,200],
                                  labels=['<18','18-25','26-35','36-50','51-65','65+'], right=False)

    if 'mental_health_history' in df.columns:
        df['has_mh_history'] = (df['mental_health_history'].str.lower() == 'yes').astype(int)
    if 'seeks_treatment' in df.columns:
        df['seeks_treatment_flag'] = (df['seeks_treatment'].str.lower() == 'yes').astype(int)

    # Encode categoricals
    mapping = {}
    for col in CATEGORICAL_COLS + ['age_group']:
        if col not in df.columns:
            continue
        df[col] = df[col].astype(str)
        enc = {v: i for i, v in enumerate(sorted(df[col].unique()))}
        df[col] = df[col].map(enc)
        mapping[col] = enc

    target_enc = {'Low': 0, 'Medium': 1, 'High': 2}
    df[TARGET] = df[TARGET].map(target_enc)
    df.dropna(subset=[TARGET], inplace=True)
    df[TARGET] = df[TARGET].astype(int)
    mapping[TARGET] = target_enc

    df.to_csv(STRUCTURED_CSV, index=False)
    with open(LABEL_MAP_JSON, 'w') as f:
        json.dump(mapping, f, indent=2)

    print(f"  Final shape: {df.shape}")
    print(f"  Class distribution:\n{df[TARGET].value_counts().rename({0:'Low',1:'Medium',2:'High'})}")
    print(f"\nSaved: {STRUCTURED_CSV}")

print("\nStructured preprocessing complete!")

## Step 6: Train the XGBoost Risk Model

This model takes numbers (age, stress level, sleep hours...) and predicts `Low / Medium / High` mental health risk.
It runs on CPU and takes only 2–5 minutes.

In [ ]:
import joblib, json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import shap, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix

RISK_MODEL_PKL  = f"{MODELS_DIR}/risk_predictor.pkl"
RISK_METRICS    = f"{MODELS_DIR}/risk_model_metrics.json"
PLOTS_DIR       = f"{MODELS_DIR}/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

TARGET     = 'mental_health_risk'
LABEL_NAMES = {0: 'Low', 1: 'Medium', 2: 'High'}

if os.path.exists(RISK_MODEL_PKL):
    print(f"Risk model already trained: {RISK_MODEL_PKL} — skipping")
else:
    print("Loading preprocessed structured data...")
    df = pd.read_csv(STRUCTURED_CSV)
    X  = df.drop(columns=[TARGET])
    y  = df[TARGET]
    print(f"  Features: {X.shape[1]}  |  Samples: {len(y):,}")
    print(f"  Class distribution: {dict(y.value_counts())}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)

    model = XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=42, n_jobs=-1
    )
    pipe = Pipeline([('scaler', StandardScaler()), ('xgb', model)])

    print("Running 5-fold cross validation...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    print(f"  CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    print("Fitting final model...")
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)
    acc    = accuracy_score(y_test, y_pred)
    roc    = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')

    print(f"\nTest Accuracy : {acc:.4f}")
    print(f"ROC-AUC (OVR) : {roc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Low','Medium','High']))

    # Save model
    joblib.dump(pipe, RISK_MODEL_PKL)

    # Save metrics
    metrics = {
        'cv_accuracy_mean': float(cv_scores.mean()),
        'cv_accuracy_std':  float(cv_scores.std()),
        'test_accuracy':    float(acc),
        'roc_auc_ovr':      float(roc),
        'feature_names':    X.columns.tolist(),
    }
    with open(RISK_METRICS, 'w') as f:
        json.dump(metrics, f, indent=2)

    # SHAP plot
    print("\nGenerating SHAP explainability plot...")
    explainer  = shap.TreeExplainer(pipe['xgb'])
    X_scaled   = pipe['scaler'].transform(X_test)
    shap_vals  = explainer.shap_values(X_scaled)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_scaled, feature_names=X.columns.tolist(),
                      plot_type='bar', show=False, class_names=['Low','Medium','High'])
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/shap_summary.png", dpi=150, bbox_inches='tight')
    plt.close()

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Low','Medium','High'], yticklabels=['Low','Medium','High'])
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.title('Mental Health Risk — Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/confusion_matrix.png", dpi=150)
    plt.close()

    print(f"\nRisk model saved: {RISK_MODEL_PKL}")

print("\nRisk model training complete!")

## Step 7: Fine-Tune Mistral-7B with QLoRA (T4-Optimized)

This is the main event — teaching Mistral-7B to be a mental health assistant.

**What QLoRA does:** Instead of retraining all 7 billion parameters (would need 80 GB VRAM),
it freezes the original model and trains tiny "adapter" matrices. The model is also
compressed to 4-bit precision. Together, this fits in **~10 GB VRAM** on a T4.

**Free T4 settings used (vs. full settings):**
| Setting | Full version | T4 free version |
|---------|-------------|------------------|
| `max_seq_length` | 2048 | **512** |
| `batch_size` | 2 | **1** |
| `epochs` | 3 | **1** |
| `lora_r` | 16 | **8** |
| `training samples` | 45k | **~20k** |

Estimated time: **1–2 hours**

In [ ]:
# Check how much VRAM we have before starting
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    if free < 10e9:
        print("WARNING: Less than 10 GB free VRAM. Consider restarting the runtime first.")
    else:
        print("Good — enough VRAM to start fine-tuning")

In [ ]:
import json, torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

LLM_OUTPUT_DIR = f"{MODELS_DIR}/mindforge-lora"

# ── Skip if already trained ───────────────────────────────────
if os.path.exists(f"{LLM_OUTPUT_DIR}/adapter_config.json"):
    print(f"LLM already fine-tuned: {LLM_OUTPUT_DIR} — skipping")
    print("Delete that folder if you want to retrain.")
else:
    os.makedirs(LLM_OUTPUT_DIR, exist_ok=True)

    # ── T4-optimized settings ─────────────────────────────────
    MAX_SEQ_LENGTH = 512     # Critical: 2048 would OOM on T4
    LORA_R         = 8       # Rank: lower = less VRAM
    LORA_ALPHA     = 16
    BATCH_SIZE     = 1
    GRAD_ACCUM     = 8       # Effective batch = 1*8 = 8
    EPOCHS         = 1
    LEARNING_RATE  = 2e-4

    # ── Load datasets ─────────────────────────────────────────
    def load_jsonl(path):
        records = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records

    print("Loading datasets...")
    train_records = load_jsonl(TRAIN_JSONL)
    val_records   = load_jsonl(VAL_JSONL)
    train_ds = Dataset.from_list(train_records)
    val_ds   = Dataset.from_list(val_records)
    print(f"  Train: {len(train_ds):,} examples")
    print(f"  Val  : {len(val_ds):,} examples")

    # ── Load base model (4-bit quantized — fits T4) ───────────
    BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
    print(f"\nLoading base model: {BASE_MODEL}")
    print("(~5-10 min first time, downloads ~4 GB from HuggingFace)")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name    = BASE_MODEL,
        max_seq_length= MAX_SEQ_LENGTH,
        dtype         = None,        # auto-detect (float16 on T4)
        load_in_4bit  = True,        # critical for T4 VRAM
    )
    print("Base model loaded!")

    # ── Attach LoRA adapters ───────────────────────────────────
    print("Attaching LoRA adapters...")
    model = FastLanguageModel.get_peft_model(
        model,
        r                    = LORA_R,
        lora_alpha           = LORA_ALPHA,
        lora_dropout         = 0.05,
        target_modules       = ["q_proj","k_proj","v_proj","o_proj",
                                 "gate_proj","up_proj","down_proj"],
        bias                 = "none",
        use_gradient_checkpointing = True,  # saves VRAM
        random_state         = 42,
    )

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")
    print("(Only ~1-2% of params are trained — that's the magic of LoRA!)")

    # ── Check VRAM after loading ───────────────────────────────
    free, total_vram = torch.cuda.mem_get_info()
    used = (total_vram - free) / 1e9
    print(f"\nVRAM used after model load: {used:.1f} GB / {total_vram/1e9:.1f} GB")

    # ── Training arguments (T4-optimized) ─────────────────────
    training_args = TrainingArguments(
        output_dir                  = LLM_OUTPUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = 50,
        learning_rate               = LEARNING_RATE,
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        fp16                        = True,    # float16 on T4
        logging_steps               = 25,
        save_steps                  = 200,     # checkpoint every 200 steps → saved to Drive
        eval_steps                  = 200,
        save_total_limit            = 2,       # keep last 2 checkpoints
        evaluation_strategy         = "steps",
        load_best_model_at_end      = True,
        report_to                   = "none",  # no wandb needed
        seed                        = 42,
    )

    # ── SFTTrainer ─────────────────────────────────────────────
    trainer = SFTTrainer(
        model             = model,
        tokenizer         = tokenizer,
        train_dataset     = train_ds,
        eval_dataset      = val_ds,
        dataset_text_field= "text",
        max_seq_length    = MAX_SEQ_LENGTH,
        args              = training_args,
        packing           = True,   # packs short sequences together → 30% faster
    )

    print("\nStarting fine-tuning...")
    print("Expected time: 1-2 hours on free T4")
    print("You will see loss decreasing every 25 steps — that means it's working!")
    print("Loss should go from ~2.0 → ~1.0 over the course of training.\n")

    trainer_stats = trainer.train()

    # ── Save LoRA adapters ────────────────────────────────────
    print(f"\nSaving LoRA adapters to {LLM_OUTPUT_DIR}...")
    model.save_pretrained(LLM_OUTPUT_DIR)
    tokenizer.save_pretrained(LLM_OUTPUT_DIR)

    train_time = trainer_stats.metrics.get('train_runtime', 0)
    final_loss = trainer_stats.metrics.get('train_loss', 'N/A')
    print(f"\nTraining complete!")
    print(f"  Time      : {train_time/60:.1f} minutes")
    print(f"  Final loss: {final_loss}")
    print("  (Loss < 1.2 is good for mental health conversations)")

## Step 8: Test the Fine-Tuned Model

Let's do a quick sanity check — ask the model a mental health question and see if it responds well.

In [ ]:
# Quick inference test
from unsloth import FastLanguageModel

print("Loading fine-tuned model for inference...")
infer_model, infer_tokenizer = FastLanguageModel.from_pretrained(
    model_name    = LLM_OUTPUT_DIR,
    max_seq_length= 512,
    dtype         = None,
    load_in_4bit  = True,
)
FastLanguageModel.for_inference(infer_model)  # enable native 2x faster inference

def ask_mindforge(question: str, max_new_tokens: int = 256) -> str:
    SYSTEM = (
        "You are MindForge, a compassionate and knowledgeable mental health AI assistant. "
        "You provide evidence-based psychological support, education, and guidance. "
        "Always remind users to seek professional help for serious concerns."
    )
    prompt = f"<s>[INST] <<SYS>>\n{SYSTEM}\n<</SYS>>\n\n{question} [/INST]"
    inputs = infer_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = infer_model.generate(
            **inputs,
            max_new_tokens    = max_new_tokens,
            temperature       = 0.7,
            top_p             = 0.9,
            repetition_penalty= 1.1,
            do_sample         = True,
        )
    full_text = infer_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response
    if "[/INST]" in full_text:
        return full_text.split("[/INST]")[-1].strip()
    return full_text

# Test questions
test_questions = [
    "I've been feeling really anxious lately and can't sleep. What should I do?",
    "What is cognitive behavioral therapy and how does it help with depression?",
]

for q in test_questions:
    print(f"\nQuestion: {q}")
    print("-" * 60)
    response = ask_mindforge(q)
    print(f"MindForge: {response}")
    print()

## Step 9: Save Everything to Google Drive

This backs up all trained models to your Drive so you don't lose them when the Colab session ends.

In [ ]:
import shutil

print("Saving trained models to Google Drive...")
print("(This may take a few minutes for the LLM adapters)\n")

def copy_to_drive(src: str, dst: str, label: str):
    if not os.path.exists(src):
        print(f"  SKIP: {label} — source not found ({src})")
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    size = sum(os.path.getsize(os.path.join(r,f))
               for r,_,files in os.walk(dst) for f in files) if os.path.isdir(dst) \
           else os.path.getsize(dst)
    print(f"  Saved {label}: {size/1e6:.0f} MB → {dst}")

# 1. LoRA adapters (the fine-tuned LLM)
copy_to_drive(
    f"{MODELS_DIR}/mindforge-lora",
    f"{DRIVE_MODELS}/mindforge-lora",
    "LLM LoRA adapters"
)

# 2. Risk model
copy_to_drive(
    f"{MODELS_DIR}/risk_predictor.pkl",
    f"{DRIVE_MODELS}/risk_predictor.pkl",
    "XGBoost risk model"
)

# 3. Metrics and plots
copy_to_drive(
    f"{MODELS_DIR}/risk_model_metrics.json",
    f"{DRIVE_MODELS}/risk_model_metrics.json",
    "Risk model metrics"
)
copy_to_drive(
    f"{MODELS_DIR}/plots",
    f"{DRIVE_MODELS}/plots",
    "SHAP + confusion matrix plots"
)

# 4. Processed data (so you don't need to re-preprocess next session)
copy_to_drive(
    f"{DATA_PROCESSED}/train.jsonl",
    f"{DRIVE_ROOT}/data/processed/train.jsonl",
    "Training JSONL"
)
copy_to_drive(
    f"{DATA_PROCESSED}/val.jsonl",
    f"{DRIVE_ROOT}/data/processed/val.jsonl",
    "Validation JSONL"
)
copy_to_drive(
    f"{DATA_PROCESSED}/structured_clean.csv",
    f"{DRIVE_ROOT}/data/processed/structured_clean.csv",
    "Structured CSV"
)
copy_to_drive(
    f"{DATA_PROCESSED}/label_mapping.json",
    f"{DRIVE_ROOT}/data/processed/label_mapping.json",
    "Label mapping"
)

print("\nAll artifacts saved to Google Drive!")
print(f"\nYour Drive folder: {DRIVE_ROOT}")
print("Structure:")
print("  MindForge/")
print("  ├── data/")
print("  │   ├── raw/        ← your 5 original datasets")
print("  │   └── processed/  ← train.jsonl, val.jsonl, structured_clean.csv")
print("  └── models/")
print("      ├── mindforge-lora/    ← fine-tuned LLM (LoRA adapters)")
print("      ├── risk_predictor.pkl ← XGBoost model")
print("      └── plots/            ← SHAP + confusion matrix")

## Done! What's Next?

### Your trained models are on Google Drive
Every time you want to use them, just run Steps 0–2 (GPU check, mount Drive, set paths), then load from Drive.

### Next step: Push to GitHub
1. Push all the **code** (not the large data/model files) to a public GitHub repo
2. Update the README to link to this notebook ("Open in Colab" badge)
3. Optionally upload the LoRA adapters to [HuggingFace Hub](https://huggingface.co) (free) for easy sharing

### Metrics to put in your README / resume:
- Check `DRIVE_MODELS/risk_model_metrics.json` for XGBoost accuracy and ROC-AUC
- Report the final LLM training loss (printed above after Step 7)
- Show the SHAP and confusion matrix plots from `DRIVE_MODELS/plots/`

> **Reminder:** MindForge is an educational project. Not a substitute for professional mental health care.